In [26]:
import pandas as pd
import random
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import roc_auc_score

In [27]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [28]:
df_train = pd.read_csv('data/train.csv', encoding="utf-8")

In [43]:
print(df_train['label'].value_counts())

label
1    5284
3    2967
2    2882
4    2743
0    2425
5    2068
Name: count, dtype: int64


In [29]:
df_test = pd.read_csv('data/test.csv', encoding="utf-8")

In [30]:
df_val = pd.read_csv('data/valid.csv', encoding="utf-8")

In [31]:
def clean(text):
    if pd.isna(text):
        return text
    text = text.lower()
    text = text.strip()
    text = re.sub(r'https\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [32]:
df_train['statement'] = df_train['statement'].apply(clean)

df_test['statement'] = df_test['statement'].apply(clean)
df_val['statement'] = df_val['statement'].apply(clean)

In [33]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# Tokenize without truncation
token_lengths = [len(tokenizer.tokenize(text)) for text in df_train['statement']]
max_length = max(token_lengths)

train_encodings = tokenizer(
    df_train['statement'].to_list(),
    padding=True,
    truncation=True,
    max_length=max_length
)
test_encodings = tokenizer(
    df_test['statement'].to_list(), 
    padding=True, 
    truncation=True, 
    max_length=max_length
)

val_encodings = tokenizer(
    df_val['statement'].to_list(),
    padding=True,
    truncation=True,
    max_length=max_length
)

In [34]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        print({k: len(v) for k, v in self.encodings.items()})
        print(len(self.labels))

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [35]:
train_dataset = MyDataset(train_encodings, df_train['label'].to_list())
test_dataset = MyDataset(test_encodings, df_test['label'].to_list())
val_dataset = MyDataset(val_encodings, df_val['label'].to_list())

In [36]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=6)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
accuracy = evaluate.load("accuracy")
auc_metric = evaluate.load("roc_auc")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)
    
    auc = roc_auc_score(
        labels,
        probs,
        multi_class="ovr"
    )

    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    
    acc = (preds == labels).mean()

    return {
        "accuracy": acc,
        "f1": f1,
        "auc": auc
    }

In [38]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/output",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=1e-5,        
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,            
    weight_decay=0.01,
    seed=SEED,
    dataloader_num_workers=0,
    report_to='none',
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [39]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

C:\master\disertatie\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\surua\AppData\Local\Temp\ipykernel_16020\1545554766.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# at the end on validation data
predictions = trainer.predict(test_dataset)

final_metrics = compute_metrics(
    (predictions.predictions, predictions.label_ids)
)

print("\nFinal Evaluation:")
print(final_metrics)